
Korean Chatbot Transformer
songys/chatbot_data.csv
============================================================

In [1]:
import os
import re
import random
import math
import pandas as pd
import numpy as np

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

------------------------------------------------------------
1. Reproducibility
------------------------------------------------------------

In [3]:
SEED = 42

In [4]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [5]:
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
print("Device:", device)

Device: cuda


------------------------------------------------------------
2. Hyperparameters
------------------------------------------------------------

In [8]:
N_LAYERS = 1
D_MODEL = 368
N_HEADS = 8
D_FF = 1024
DROPOUT = 0.2

In [9]:
WARMUP_STEPS = 1000
BATCH_SIZE = 64
EPOCHS = 20

In [10]:
MAX_LEN = 40

In [11]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
BOS_TOKEN = "<start>"
EOS_TOKEN = "<end>"

In [12]:
SPECIAL_TOKENS = [
    PAD_TOKEN,
    UNK_TOKEN,
    BOS_TOKEN,
    EOS_TOKEN
]

------------------------------------------------------------
3. Load Dataset
------------------------------------------------------------

In [13]:
# songys/Chatbot_data 저장소의 원본 CSV를 스크립트 폴더에 저장
import urllib.request

In [94]:
DATA_URL = "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv"
CANDIDATE_DIRS = [
    os.getcwd(),
    os.path.join(os.path.expanduser("~"), "Downloads", "transformer"),
    "/Users/robotjang/Downloads/transformer"
]
SCRIPT_DIR = next(
    (
        path for path in CANDIDATE_DIRS
        if os.path.exists(os.path.join(path, "ChatbotData.csv"))
        and os.path.exists(os.path.join(path, "ko.bin"))
    ),
    next(
        (
            path for path in CANDIDATE_DIRS
            if os.path.exists(os.path.join(path, "ChatbotData.csv"))
        ),
        os.getcwd()
    )
)
DATA_PATH = os.path.join(SCRIPT_DIR, "ChatbotData.csv")
print("Project folder:", SCRIPT_DIR)

Project folder: /content


In [16]:
if not os.path.exists(DATA_PATH):
    print("Downloading dataset from GitHub...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

In [88]:
df = pd.read_csv(DATA_PATH)

In [18]:
print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (11823, 3)
                 Q            A  label
0           12시 땡!   하루가 또 가네요.      0
1      1지망 학교 떨어졌어    위로해 드립니다.      0
2     3박4일 놀러가고 싶다  여행은 언제나 좋죠.      0
3  3박4일 정도 놀러가고 싶다  여행은 언제나 좋죠.      0
4          PPL 심하네   눈살이 찌푸려지죠.      0


------------------------------------------------------------
4. Column detection
------------------------------------------------------------

songys/chatbot_data.csv는 일반적으로
Q : 질문
A : 답변
label : 분류

컬럼명이 다른 경우 자동 대응

In [78]:
if "Q" in df.columns and "A" in df.columns:
    question_col = "Q"
    answer_col = "A"
elif "question" in df.columns and "answer" in df.columns:
    question_col = "question"
    answer_col = "answer"
else:
    question_col = df.columns[0]
    answer_col = df.columns[1]

In [20]:
print("Question column:", question_col)
print("Answer column:", answer_col)

Question column: Q
Answer column: A


------------------------------------------------------------
5. Text Cleaning
------------------------------------------------------------

In [79]:
def clean_text(text):
    text = str(text)

    # 한글, 영문, 숫자, 기본 문장부호만 유지
    text = re.sub(r"[^가-힣a-zA-Z0-9\s.,!?~]", " ", text)

    # 연속 공백 제거
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [80]:
df[question_col] = df[question_col].astype(str).apply(clean_text)
df[answer_col] = df[answer_col].astype(str).apply(clean_text)

In [81]:
# 빈 문장 제거
df = df[
    (df[question_col].str.len() > 0) &
    (df[answer_col].str.len() > 0)
].reset_index(drop=True)

In [23]:
print("Cleaned dataset:", len(df))

Cleaned dataset: 11823


------------------------------------------------------------
6. Data Augmentation
------------------------------------------------------------

In [92]:
WORD2VEC_PATH = os.path.join(SCRIPT_DIR, "ko.bin")
word2vec_model = None

In [93]:
if os.path.exists(WORD2VEC_PATH):
    try:
        from gensim.models import KeyedVectors, Word2Vec
        from gensim.utils import SaveLoad

        try:
            word2vec_model = Word2Vec.load(WORD2VEC_PATH).wv
        except Exception:
            try:
                original_load_specials = Word2Vec._load_specials
                Word2Vec._load_specials = SaveLoad._load_specials
                try:
                    legacy_model = Word2Vec.load(WORD2VEC_PATH, mmap=None)
                finally:
                    Word2Vec._load_specials = original_load_specials

                word2vec_model = KeyedVectors(
                    vector_size=legacy_model.vector_size
                )
                word2vec_model.index_to_key = list(legacy_model.index2word)
                word2vec_model.key_to_index = {
                    word: index
                    for index, word in enumerate(word2vec_model.index_to_key)
                }
                word2vec_model.vectors = legacy_model.syn0
            except Exception:
                word2vec_model = KeyedVectors.load_word2vec_format(
                    WORD2VEC_PATH,
                    binary=True
                )
        print("Loaded Word2Vec:", WORD2VEC_PATH)
    except Exception as error:
        print("Word2Vec disabled:", error)
else:
    print("Word2Vec not found; using noise injection only:", WORD2VEC_PATH)

Word2Vec not found; using noise injection only: /content/ko.bin


In [27]:
def lexicon_substitution(text, probability=0.25):
    if word2vec_model is None:
        return text

    words = text.split()
    substituted = []

    for word in words:
        lookup_word = word.strip(".,!?~")

        if (
            random.random() < probability and
            lookup_word in word2vec_model.key_to_index
        ):
            candidates = word2vec_model.most_similar(
                lookup_word,
                topn=10
            )
            candidates = [
                candidate
                for candidate, similarity in candidates
                if candidate != lookup_word and " " not in candidate
            ]

            if candidates:
                replacement = random.choice(candidates)
                suffix = word[len(lookup_word):]
                word = replacement + suffix

        substituted.append(word)

    return " ".join(substituted)

In [28]:
def noise_injection(text):
    words = text.split()
    if len(words) > 2:
        idx = random.randint(0, len(words) - 2)
        words[idx], words[idx+1] = words[idx+1], words[idx]
    return " ".join(words)

In [29]:
def augment_data(df, question_col, answer_col, target_size=30000):
    augmented_list = []
    current_size = len(df)
    
    while len(augmented_list) + current_size < target_size:
        idx = random.randint(0, current_size - 1)
        q = df.iloc[idx][question_col]
        a = df.iloc[idx][answer_col]
        
        # 유사어 치환 후 일부 샘플에 단어 순서 noise 적용
        aug_q = lexicon_substitution(q)
        if random.random() < 0.5:
            aug_q = noise_injection(aug_q)
        augmented_list.append({question_col: aug_q, answer_col: a})
        
    return pd.concat([df, pd.DataFrame(augmented_list)], ignore_index=True)

In [75]:
# 원본 데이터는 먼저 train/validation으로 분리하고,
# validation에는 augmentation을 적용하지 않습니다.
print("Original dataset size:", len(df))

Original dataset size: 30000


------------------------------------------------------------
7. Train / Validation Split
------------------------------------------------------------

In [82]:
indices = np.arange(len(df))
np.random.shuffle(indices)

In [83]:
split = int(len(indices) * 0.9)

In [84]:
train_indices = indices[:split]
valid_indices = indices[split:]

In [85]:
train_df = df.iloc[train_indices].reset_index(drop=True)
valid_df = df.iloc[valid_indices].reset_index(drop=True)

In [86]:
# train에만 augmentation을 적용해 validation 데이터 누수를 방지합니다.
train_df = augment_data(train_df, question_col, answer_col, target_size=30000)

print("Train:", len(train_df))
print("Validation:", len(valid_df))

Train: 30000
Validation: 1183


------------------------------------------------------------
8. Tokenizer
------------------------------------------------------------

In [36]:
def tokenize(text):
    # 간단한 한국어 whitespace tokenizer
    return text.split()

------------------------------------------------------------
9. Vocabulary
------------------------------------------------------------

In [37]:
counter = {}

In [38]:
for text in train_df[question_col]:
    for token in tokenize(text):
        counter[token] = counter.get(token, 0) + 1

In [40]:
for text in train_df[answer_col]:
    for token in tokenize(text):
        counter[token] = counter.get(token, 0) + 1

In [41]:
# 빈도순 정렬
tokens = sorted(
    counter.items(),
    key=lambda x: x[1],
    reverse=True
)

In [42]:
itos = SPECIAL_TOKENS.copy()

In [43]:
for token, freq in tokens:
    if token not in itos:
        itos.append(token)

In [44]:
stoi = {
    token: idx
    for idx, token in enumerate(itos)
}

In [45]:
PAD_IDX = stoi[PAD_TOKEN]
UNK_IDX = stoi[UNK_TOKEN]
BOS_IDX = stoi[BOS_TOKEN]
EOS_IDX = stoi[EOS_TOKEN]

In [46]:
VOCAB_SIZE = len(itos)

In [47]:
print("Vocabulary size:", VOCAB_SIZE)

Vocabulary size: 21527


------------------------------------------------------------
10. Numericalization
------------------------------------------------------------

In [48]:
def encode(text):
    tokens = tokenize(text)

    ids = [BOS_IDX]

    for token in tokens[:MAX_LEN - 2]:
        ids.append(stoi.get(token, UNK_IDX))

    ids.append(EOS_IDX)

    return ids

------------------------------------------------------------
11. Dataset
------------------------------------------------------------

In [49]:
class ChatbotDataset(Dataset):

    def __init__(self, dataframe):
        self.data = dataframe

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        question = self.data.iloc[idx][question_col]
        answer = self.data.iloc[idx][answer_col]

        src = encode(question)
        tgt = encode(answer)

        return torch.tensor(src), torch.tensor(tgt)

------------------------------------------------------------
12. Padding Collate
------------------------------------------------------------

In [50]:
def collate_fn(batch):

    src_batch = []
    tgt_batch = []

    for src, tgt in batch:
        src_batch.append(src)
        tgt_batch.append(tgt)

    src_batch = nn.utils.rnn.pad_sequence(
        src_batch,
        batch_first=True,
        padding_value=PAD_IDX
    )

    tgt_batch = nn.utils.rnn.pad_sequence(
        tgt_batch,
        batch_first=True,
        padding_value=PAD_IDX
    )

    return src_batch, tgt_batch

In [51]:
train_dataset = ChatbotDataset(train_df)
valid_dataset = ChatbotDataset(valid_df)

In [52]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

In [53]:
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

------------------------------------------------------------
13. Positional Encoding
------------------------------------------------------------

In [54]:
class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        dropout=0.1,
        max_len=5000
    ):
        super().__init__()

        self.dropout = nn.Dropout(dropout)

        position = torch.arange(
            max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ) * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_len,
            d_model
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        x = x + self.pe[:, :x.size(1)]

        return self.dropout(x)

------------------------------------------------------------
14. Transformer Seq2Seq Model
------------------------------------------------------------

In [55]:
class TransformerChatbot(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=368,
        n_heads=8,
        n_layers=1,
        d_ff=1024,
        dropout=0.2
    ):
        super().__init__()

        self.d_model = d_model

        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_IDX
        )

        self.positional_encoding = PositionalEncoding(
            d_model,
            dropout
        )

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=n_heads,
            num_encoder_layers=n_layers,
            num_decoder_layers=n_layers,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True
        )

        self.output_layer = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(
        self,
        src,
        tgt,
        src_key_padding_mask=None,
        tgt_key_padding_mask=None,
        tgt_mask=None
    ):

        src_emb = self.embedding(src) * math.sqrt(self.d_model)
        tgt_emb = self.embedding(tgt) * math.sqrt(self.d_model)

        src_emb = self.positional_encoding(src_emb)
        tgt_emb = self.positional_encoding(tgt_emb)

        output = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )

        return self.output_layer(output)

In [56]:
model = TransformerChatbot(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT
).to(device)

In [57]:
print(model)

TransformerChatbot(
  (embedding): Embedding(21527, 368, padding_idx=0)
  (positional_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.2, inplace=False)
  )
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0): TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=368, out_features=368, bias=True)
          )
          (linear1): Linear(in_features=368, out_features=1024, bias=True)
          (dropout): Dropout(p=0.2, inplace=False)
          (linear2): Linear(in_features=1024, out_features=368, bias=True)
          (norm1): LayerNorm((368,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((368,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.2, inplace=False)
          (dropout2): Dropout(p=0.2, inplace=False)
        )
      )
      (norm): LayerNorm((368,), eps=1e-05, elementwise_affine=True)
    )
   

------------------------------------------------------------
15. Causal Mask
------------------------------------------------------------

In [58]:
def generate_square_subsequent_mask(size):

    mask = torch.triu(
        torch.ones(
            size,
            size,
            device=device
        ),
        diagonal=1
    )

    mask = mask.masked_fill(
        mask == 1,
        float("-inf")
    )

    return mask

------------------------------------------------------------
16. Optimizer
------------------------------------------------------------

In [59]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1.0,
    betas=(0.9, 0.98),
    eps=1e-9
)

------------------------------------------------------------
17. Noam / Transformer Warmup Schedule
------------------------------------------------------------

In [60]:
def transformer_lr(step):

    step = max(step, 1)

    return (
        D_MODEL ** -0.5
        * min(
            step ** -0.5,
            step * WARMUP_STEPS ** -1.5
        )
    )

In [61]:
def update_learning_rate(step):

    lr = transformer_lr(step)

    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    return lr

------------------------------------------------------------
18. Loss
------------------------------------------------------------

In [62]:
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

------------------------------------------------------------
19. Training
------------------------------------------------------------

In [63]:
def train_one_epoch():

    model.train()

    total_loss = 0

    for src, tgt in train_loader:

        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        tgt_mask = generate_square_subsequent_mask(
            tgt_input.size(1)
        )

        src_padding_mask = (
            src == PAD_IDX
        )

        tgt_padding_mask = (
            tgt_input == PAD_IDX
        )

        optimizer.zero_grad()

        logits = model(
            src,
            tgt_input,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            tgt_mask=tgt_mask
        )

        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            tgt_output.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        global global_step

        global_step += 1

        update_learning_rate(global_step)

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

------------------------------------------------------------
20. Validation
------------------------------------------------------------

In [64]:
@torch.no_grad()
def evaluate():

    model.eval()

    total_loss = 0

    for src, tgt in valid_loader:

        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        tgt_mask = generate_square_subsequent_mask(
            tgt_input.size(1)
        )

        src_padding_mask = (
            src == PAD_IDX
        )

        tgt_padding_mask = (
            tgt_input == PAD_IDX
        )

        logits = model(
            src,
            tgt_input,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            tgt_mask=tgt_mask
        )

        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            tgt_output.reshape(-1)
        )

        total_loss += loss.item()

    return total_loss / len(valid_loader)

------------------------------------------------------------
21. Training Loop
------------------------------------------------------------

In [65]:
global_step = 0

In [66]:
best_valid_loss = float("inf")

In [67]:
for epoch in range(1, EPOCHS + 1):

    train_loss = train_one_epoch()
    valid_loss = evaluate()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.8f}"
    )

    if valid_loss < best_valid_loss:

        best_valid_loss = valid_loss

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "stoi": stoi,
                "itos": itos,
                "vocab_size": VOCAB_SIZE,
                "hyperparameters": {
                    "n_layers": N_LAYERS,
                    "d_model": D_MODEL,
                    "n_heads": N_HEADS,
                    "d_ff": D_FF,
                    "dropout": DROPOUT,
                    "warmup_steps": WARMUP_STEPS,
                    "batch_size": BATCH_SIZE,
                    "epochs": EPOCHS
                }
            },
            "korean_chatbot_transformer.pt"
        )

        print("  -> Best model saved")

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 01/20 | Train Loss: 7.0499 | Valid Loss: 5.8360 | LR: 0.00069565
  -> Best model saved
Epoch 02/20 | Train Loss: 5.1276 | Valid Loss: 4.2756 | LR: 0.00139129
  -> Best model saved
Epoch 03/20 | Train Loss: 3.9330 | Valid Loss: 3.6185 | LR: 0.00146507
  -> Best model saved
Epoch 04/20 | Train Loss: 3.3930 | Valid Loss: 3.3082 | LR: 0.00126879
  -> Best model saved
Epoch 05/20 | Train Loss: 3.0953 | Valid Loss: 3.0869 | LR: 0.00113484
  -> Best model saved
Epoch 06/20 | Train Loss: 2.8707 | Valid Loss: 2.8775 | LR: 0.00103596
  -> Best model saved
Epoch 07/20 | Train Loss: 2.6880 | Valid Loss: 2.6867 | LR: 0.00095912
  -> Best model saved
Epoch 08/20 | Train Loss: 2.5420 | Valid Loss: 2.5699 | LR: 0.00089717
  -> Best model saved
Epoch 09/20 | Train Loss: 2.4098 | Valid Loss: 2.4685 | LR: 0.00084586
  -> Best model saved
Epoch 10/20 | Train Loss: 2.2836 | Valid Loss: 2.3313 | LR: 0.00080245
  -> Best model saved
Epoch 11/20 | Train Loss: 2.1721 | Valid Loss: 2.1913 | LR: 0.00076511

------------------------------------------------------------
22. Load Best Model
------------------------------------------------------------

In [68]:
checkpoint = torch.load(
    "korean_chatbot_transformer.pt",
    map_location=device
)

In [69]:
model.load_state_dict(
    checkpoint["model_state_dict"]
)

<All keys matched successfully>

In [70]:
print("Best model loaded.")

Best model loaded.


------------------------------------------------------------
23. Greedy Decoding
------------------------------------------------------------

In [71]:
@torch.no_grad()
def generate_response(
    sentence,
    max_len=MAX_LEN
):

    model.eval()

    src = torch.tensor(
        [encode(sentence)],
        dtype=torch.long,
        device=device
    )

    src_padding_mask = (
        src == PAD_IDX
    )

    ys = torch.tensor(
        [[BOS_IDX]],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_len):

        tgt_mask = generate_square_subsequent_mask(
            ys.size(1)
        )

        tgt_padding_mask = (
            ys == PAD_IDX
        )

        output = model(
            src,
            ys,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            tgt_mask=tgt_mask
        )

        next_token = output[:, -1, :].argmax(
            dim=-1
        ).item()

        ys = torch.cat(
            [
                ys,
                torch.tensor(
                    [[next_token]],
                    device=device
                )
            ],
            dim=1
        )

        if next_token == EOS_IDX:
            break

    result = []

    for idx in ys[0].tolist()[1:]:

        if idx in [
            EOS_IDX,
            PAD_IDX
        ]:
            break

        result.append(
            itos[idx]
        )

    return " ".join(result)

------------------------------------------------------------
24. Test
------------------------------------------------------------

In [72]:
test_sentences = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야."
]

In [73]:
print("\n===== Chatbot Test =====")


===== Chatbot Test =====


In [74]:
for sentence in test_sentences:

    response = generate_response(sentence)

    print(f"\n입력 : {sentence}")
    print(f"응답 : {response}")


입력 : 지루하다, 놀러가고 싶어.
응답 : 돈은 쓴만큼 또 생긴다고 하던데요.

입력 : 오늘 일찍 일어났더니 피곤하다.
응답 : 마스크 착용 하시고 외출하세요.

입력 : 간만에 여자친구랑 데이트 하기로 했어.
응답 : 마음에 드는 책을 잘 찾아보세요.

입력 : 집에 있는다는 소리야.
응답 : 사랑의 빈자리인가봐요.


In [95]:
from collections import Counter


def ngram_counts(tokens, n):
    return Counter(
        tuple(tokens[index:index + n])
        for index in range(len(tokens) - n + 1)
    )


def sentence_bleu(reference, hypothesis, max_n=4):
    reference_tokens = tokenize(reference)
    hypothesis_tokens = tokenize(hypothesis)

    if not hypothesis_tokens:
        return 0.0

    precisions = []
    for n in range(1, max_n + 1):
        candidate_counts = ngram_counts(hypothesis_tokens, n)
        reference_counts = ngram_counts(reference_tokens, n)
        total = sum(candidate_counts.values())

        if total == 0:
            precisions.append(1e-9)
            continue

        clipped = sum(
            min(count, reference_counts[ngram])
            for ngram, count in candidate_counts.items()
        )
        precisions.append(max(clipped / total, 1e-9))

    geometric_mean = math.exp(
        sum(math.log(precision) for precision in precisions) / max_n
    )
    brevity_penalty = min(
        1.0,
        math.exp(1 - len(reference_tokens) / len(hypothesis_tokens))
    )
    return brevity_penalty * geometric_mean


sample_count = min(100, len(valid_df))
bleu_scores = {1: [], 2: [], 4: []}
response_examples = []

for index in range(sample_count):
    question = valid_df.iloc[index][question_col]
    reference = valid_df.iloc[index][answer_col]
    hypothesis = generate_response(question)

    for n in bleu_scores:
        bleu_scores[n].append(sentence_bleu(reference, hypothesis, max_n=n))

    if index < 10:
        response_examples.append((question, reference, hypothesis))

print("===== Validation Response Examples =====")
for question, reference, hypothesis in response_examples:
    print(f"\n질문: {question}")
    print(f"참조 답변: {reference}")
    print(f"생성 답변: {hypothesis}")

print("\n===== BLEU Scores =====")
for n, scores in bleu_scores.items():
    print(f"BLEU-{n}: {sum(scores) / len(scores):.4f}")

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


===== Validation Response Examples =====

질문: 거래처 직원을 좋아하게 됐는데 어떻게 연락할까.
참조 답변: 관계를 지속해도 되는 사이라면 직접 물어보는게 좋겠어요.
생성 답변: 지금도 충분히 잘 하고 있어요.

질문: 인턴 합격
참조 답변: 좋은 사회생활의 밑거름이 되길 바랍니다.
생성 답변: 좋은 사회생활의 밑거름이 되길 바랍니다.

질문: 어제 만났네.
참조 답변: 좋은 만남이었길 바라요.
생성 답변: 많이 힘들었겠어요.

질문: 아기 좋아하는 남자 어때?
참조 답변: 착할 거 같아요.
생성 답변: 따뜻할 거 같아요.

질문: 마지막 그녀를 만났어
참조 답변: 복잡했던 마음이 정리되길 바랄게요.
생성 답변: 꿩 먹고 알 먹고네요.

질문: 최악
참조 답변: 똥차 가고 벤츠 올 거예요.
생성 답변: 똥차 가고 벤츠 올 거예요.

질문: 이성으로는 감당 할 수 없을 것 같아
참조 답변: 잠깐 눈을 붙이고 마음을 가라앉혀보세요.
생성 답변: 이럴 때 잘 쉬는 게 중요해요.

질문: 친구 결혼식에서 축가 준비 중
참조 답변: 친구가 좋아하겠어요.
생성 답변: 친구가 좋아하겠어요.

질문: 이러다 또 흐지부지하고 말겠지
참조 답변: 이번에는 안 그러면 돼요.
생성 답변: 이번에는 안 그러면 돼요.

질문: 이 놈의 신용카드 다 없애야지
참조 답변: 소비 조절을 못하면 없애는 것도 방법이 될 거예요.
생성 답변: 소비 조절을 못하면 없애는 것도 방법이 될 거예요.

===== BLEU Scores =====
BLEU-1: 0.3804
BLEU-2: 0.3249
BLEU-4: 0.1766


## 제출용 평가 및 해석

### 1. 전처리와 데이터 증강
- `ChatbotData.csv`의 질문과 답변을 불러온 뒤 빈 문장을 제거했습니다.
- 허용 문자 정규화와 연속 공백 제거를 적용했습니다.
- `ko.bin` Word2Vec 유사어 치환과 단어 순서 noise injection을 사용했습니다.
- 원본 데이터를 먼저 train/validation으로 분리하고, train에만 증강을 적용해 validation 누수를 방지했습니다.
- train 데이터는 약 30,000개로 확장하고 validation은 원본 데이터로 유지했습니다.

### 2. 과적합 방지 하이퍼파라미터
- `DROPOUT = 0.2`: Transformer 내부 dropout으로 co-adaptation을 완화합니다.
- `EPOCHS = 20`: 매 epoch validation loss를 측정하고 최저 validation loss 모델만 저장합니다.
- `clip_grad_norm_(..., 1.0)`: gradient 폭주를 제한합니다.
- Noam warmup learning-rate schedule과 `Adam(beta1=0.9, beta2=0.98)`를 사용합니다.
- `PAD` 토큰은 loss에서 제외하고, causal mask로 미래 토큰을 보지 못하게 합니다.

### 3. 응답 사례와 BLEU
아래 평가 셀은 validation 질문에 대한 생성 응답, 참조 답변, BLEU-1/2/4를 함께 출력합니다. 대화형 답변은 정답 표현이 여러 가지일 수 있으므로 BLEU만으로 품질을 단정하지 않고, 예시 응답과 validation loss를 함께 해석합니다.

## 제출용 모델 훈련 설정

모델을 훈련하고 다음 하이퍼파라미터 및 학습 설정을 사용했습니다.

```text
Hyperparameters > n_layers: 1 > d_model: 368 > n_heads: 8 > d_ff: 1024 > dropout: 0.2
Training Parameters > Warmup Steps: 1000 > Batch Size: 64 > Epoch At: 20
```

- Optimizer: Adam
- Adam betas: `(0.9, 0.98)`
- Gradient clipping: `1.0`
- Maximum sequence length: `40`
- Dataset augmentation target: `30,000` training samples
- Augmentation: Word2Vec lexicon substitution and noise injection
- Best model selection: lowest validation loss
- Device: CUDA if available, otherwise CPU